## 🏗️ The System Architecture

To map multiple images to multiple historical text reports per patient (as structured in your CSV), you need an encoder-decoder setup:

1. **Vision Encoder:** A convolutional network (like ResNet or DenseNet) or a Vision Transformer (ViT) that extracts deep spatial features from the chest X-rays.
2. **Text Decoder:** A causal language model (like GPT-2, DistilGPT2, or a lightweight LLaMA variant) that takes the image features and autoregressively generates the clinical report text tokens.

```
[ Input Images ] ───► [ Vision Encoder (e.g., ViT) ] ───┐
                                                         ▼
[ Ground Truth ] ───► [ Text Decoder (e.g., GPT-2) ] ──► [ Generated Report ]

```

In [1]:
import os
import ast
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
from torchvision import transforms, models
from transformers import GPT2LMHeadModel, GPT2Config, AutoTokenizer
from tqdm import tqdm
from pathlib import Path
import ast

## 🛠️ Data Preprocessing & Custom Dataset

Because your CSV stores paths and text as string representations of lists, your PyTorch `Dataset` needs to parse these strings, load the actual image files, tokenize the reports, and align them.

Here is how you can build the data pipeline:

In [2]:
class MimicReportDataset(Dataset):
    def __init__(self, csv_file, img_root_dir, tokenizer, transform=None, max_length=128):
        """
        Args:
            csv_file (str): Path to your processed CSV file.
            img_root_dir (str): Root directory where the 'files/' folder is located.
            tokenizer: Hugging Face tokenizer (e.g., GPT2Tokenizer).
        """
        self.df = pd.read_csv(csv_file)
        self.img_root_dir = img_root_dir
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # Standard image transformations for DenseNet121
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 1. Parse string representation of lists safely using abstract syntax trees
        img_paths = ast.literal_eval(row['image'])
        reports = ast.literal_eval(row['text'])
        
        # 2. Extract the first available image and its matching clinical report text
        img_path = os.path.join(self.img_root_dir, img_paths[0])
        report_text = reports[0] if len(reports) > 0 else "Findings: Normal study."
        
        # 3. Load and transform image file safely
        try:
            image = Image.open(img_path).convert('RGB')
            image = self.transform(image)
        except Exception as e:
            # Fallback tensor if a file is missing or corrupted during runtime
            image = torch.zeros(3, 224, 224)
            
        # 4. Tokenize report text for GPT-2
        tokens = self.tokenizer(
            report_text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        input_ids = tokens['input_ids'].squeeze(0)
        attention_mask = tokens['attention_mask'].squeeze(0)
        
        # Create shifting target labels for text generation loss calculation
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'image': image,
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }

## 🏗️ Define the Multimodal Model

This model extracts features from the image, projects them into the text dimension, and prepends them as visual "tokens" right before the actual report text tokens.

In [3]:
class MedicalReportGenerator(nn.Module):
    def __init__(self, gpt2_model_name="gpt2", embed_dim=768):
        super(MedicalReportGenerator, self).__init__()
        
        # 1. Vision Encoder: Using pre-trained DenseNet121
        # Excellent for chest X-rays because of feature reuse dense connections
        self.encoder = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        num_ftrs = self.encoder.classifier.in_features
        self.encoder.classifier = nn.Identity() # Remove the final classification layer
        
        # 2. Projection Layer: Maps vision features (1024) to Text Embeddings (768)
        self.projector = nn.Linear(num_ftrs, embed_dim)
        
        # 3. Text Decoder: Causal Language Model
        self.decoder = GPT2LMHeadModel.from_pretrained(gpt2_model_name)
        
    def forward(self, images, input_ids, attention_mask):
        # Step 1: Extract visual features -> Shape: [batch_size, 1024]
        visual_features = self.encoder(images)
        
        # Step 2: Project to text embedding space -> Shape: [batch_size, 768]
        image_embeddings = self.projector(visual_features)
        # Add a pseudo-sequence dimension -> Shape: [batch_size, 1, 768]
        image_embeddings = image_embeddings.unsqueeze(1) 
        
        # Step 3: Get standard text token embeddings -> Shape: [batch_size, seq_len, 768]
        text_embeddings = self.decoder.transformer.wte(input_ids)
        
        # Step 4: Concatenate visual "tokens" with actual text tokens
        # Shape becomes: [batch_size, 1 + seq_len, 768]
        multimodal_embeddings = torch.cat((image_embeddings, text_embeddings), dim=1)
        
        # Adjust attention mask to account for the extra visual token at the start
        batch_size = images.size(0)
        extra_mask = torch.ones((batch_size, 1), device=images.device)
        extended_attention_mask = torch.cat((extra_mask, attention_mask), dim=1)
        
        # Step 5: Pass through GPT-2 layers
        outputs = self.decoder(inputs_embeds=multimodal_embeddings, attention_mask=extended_attention_mask)
        return outputs.logits

## 🏎️ The Efficient Strategy: Progressive Unfreezing

To train this efficiently without destroying the pre-trained weights (and saving hours of compute), we will freeze the encoder first, train the mapping, and unfreeze later.


In [4]:
def get_trainable_parameters(model, phase="warmup"):
    if phase == "warmup":
        print("🔥 Phase 1: Freezing Encoder. Training Projector and Decoder Only.")
        # Freeze DenseNet
        for param in model.encoder.parameters():
            param.requires_grad = False
        # Keep Projector and GPT-2 Trainable
        for param in model.projector.parameters():
            param.requires_grad = True
        for param in model.decoder.parameters():
            param.requires_grad = True
            
    elif phase == "joint":
        print("🔓 Phase 2: Unfreezing Everything for joint fine-tuning.")
        for param in model.parameters():
            param.requires_grad = True
            
    return filter(lambda p: p.requires_grad, model.parameters())

## 🔄 PyTorch Training Loop Execution

This standard PyTorch loop tracks training by passing shifted tokens into the standard Cross-Entropy loss layer—meaning the model constantly tries to predict the next word in the findings section based on the image features it calculated.


In [5]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    
    for batch in dataloader:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        # Forward Pass
        logits = model(images, input_ids, attention_mask)
        
        # Real-time shifting for standard Autoregressive Generation Loss
        # We target predictions on text tokens (ignoring the visual prefix token projection)
        shift_logits = logits[:, 1:-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        
        # Flatten tensors for CrossEntropy Loss calculation
        loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(dataloader)

## 🔮 Generation Pipeline (Inference)

When running inference on new X-ray images, the model uses an autoregressive decoding loop to generate word-by-word descriptions until it hits an `<EOS>` (end-of-sentence) token.

In [6]:
def generate_report(model, image, tokenizer, device, max_len=64):
    model.eval()
    with torch.no_grad():
        image = image.to(device).unsqueeze(0) # Add batch dimension
        
        # Initialize text generation with the Start of Text Token (BOS)
        generated = torch.tensor([[tokenizer.bos_token_id]]).to(device)
        attention_mask = torch.ones((1, 1)).to(device)
        
        for _ in range(max_len):
            logits = model(image, generated, attention_mask)
            next_token_logits = logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)
            
            # Append predicted token to sequence
            generated = torch.cat((generated, next_token), dim=1)
            attention_mask = torch.cat((attention_mask, torch.ones((1, 1)).to(device)), dim=1)
            
            # Break loop early if model outputs End of Text Token
            if next_token.item() == tokenizer.eos_token_id:
                break
                
        return tokenizer.decode(generated[0], skip_special_tokens=True)



## 🛠️ Execution Checklist to Run This Code:

1. **Instantiation:** 

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MedicalReportGenerator().to(device)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 163MB/s] 


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

2. **Warmup Phase (approx. 3-5 epochs):** Run `get_trainable_parameters(model, phase="warmup")` with a learning rate of `1e-4`. This aligns your projection layer quickly.
3. **Joint Phase (approx. 5 epochs):** Run `get_trainable_parameters(model, phase="joint")` with a very low learning rate (`1e-5`) to optimize details across the entire network architecture concurrently.

## 🛠️ Connect the Dataset to Your Local Paths

Paste this cell into your notebook right after your model definition. We will point the `img_root_dir` exactly to where your `files` folder is hosted.

In [8]:
# 1. Initialize the tokenizer (matches our GPT-2 decoder)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# 2. Define the absolute or relative paths based on your workspace
# Your CSV files are right next to the notebook, and the images are inside the subfolder
IMG_ROOT = "/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/official_data_iccv_final"
TRAIN_CSV = "/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/mimic_cxr_aug_train.csv"
VALID_CSV = "/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/mimic_cxr_aug_validate.csv"

# 3. Create Train and Validation Datasets
# (Uses the MimicReportDataset class we created earlier)
train_dataset = MimicReportDataset(
    csv_file=TRAIN_CSV,
    img_root_dir=IMG_ROOT,
    tokenizer=tokenizer,
    max_length=128
)

val_dataset = MimicReportDataset(
    csv_file=VALID_CSV,
    img_root_dir=IMG_ROOT,
    tokenizer=tokenizer,
    max_length=128
)

# 4. Create DataLoaders
# Pin memory speeds up tensor transfer to your GPU/MPS backend
BATCH_SIZE = 8  # Adjust based on your available VRAM

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(size_check := f"Data pipelines ready! Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Data pipelines ready! Train batches: 8074 | Val batches: 63


## 🛠️ Step 2: Test the Pipeline with a Single Sample

Before running a heavy training loop, it is critical to run a sanity check. This verifies that Python is reading your `.jpg` paths correctly out of the directory structure and tokenizing the text without throwing any path mismatches.

In [9]:
# Pull a single batch from your train loader
try:
    sample_batch = next(iter(train_loader))
    print("✅ Success! Images and text loaded perfectly.")
    print("Image Tensor Shape:", sample_batch['image'].shape)        # Expected: [8, 3, 224, 224]
    print("Input IDs Tensor Shape:", sample_batch['input_ids'].shape) # Expected: [8, 128]
except Exception as e:
    print("❌ There is an issue with pathing or parsing:")
    print(str(e))

✅ Success! Images and text loaded perfectly.
Image Tensor Shape: torch.Size([8, 3, 224, 224])
Input IDs Tensor Shape: torch.Size([8, 128])


## 🛠️ Step 3: Initialize the Warmup Phase Training

Once your sample batch passes without an error, you can initialize the optimizer and run your first training epoch. Because you're working in a `.ipynb` notebook, let's wrap the training execution nicely using the `tqdm` progress bar you verified earlier.

In [10]:
# 1. Force the engine to target your dedicated NVIDIA GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Active Engine: {device.type.upper()} (Using NVIDIA Graphics Card)")

# 2. Re-initialize your setup components
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# 3. Instantiate the exact same model structure
model = MedicalReportGenerator().to(device)

# 4. Load your checkpoint data back into system memory
CHECKPOINT_PATH = "/kaggle/input/models/muhammad0nur/mimic-vlm-epoch1-checkpoint-pt/pytorch/default/1/mimic_vlm_epoch1_checkpoint.pt"
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

# 5. Restore the exact weights from your Epoch 1 run
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Restored weights from your previous run (Loss: {checkpoint['loss']})")

# 6. Set up the optimizer and pass it the loaded optimizer parameters
trainable_params = get_trainable_parameters(model, phase="warmup")
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4)
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Define start point for your next loop
start_epoch = checkpoint['epoch'] + 1  # This will be Epoch 2

🚀 Active Engine: CUDA (Using NVIDIA Graphics Card)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Restored weights from your previous run (Loss: 1.1568)
🔥 Phase 1: Freezing Encoder. Training Projector and Decoder Only.


In [11]:
criterion = nn.CrossEntropyLoss()
TOTAL_EPOCHS = 3  # Set how many total epochs you want to train for

for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
    model.train()
    epoch_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Training Epoch {epoch}")
    
    for batch in progress_bar:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass through your VLM pipeline
        logits = model(images, input_ids, attention_mask)
        
        # Shift tokens for autoregressive language prediction loss
        shift_logits = logits[:, 1:-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        
        loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        progress_bar.set_postfix({"batch_loss": f"{loss.item():.4f}"})
        
    print(f"✨ Epoch {epoch} Completed! Avg Loss: {epoch_loss / len(train_loader):.4f}")
    
    # Save a fresh checkpoint at the end of each new epoch
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': epoch_loss / len(train_loader)
    }, f"mimic_vlm_epoch{epoch}_checkpoint.pt")

Training Epoch 2: 100%|██████████| 8074/8074 [1:06:53<00:00,  2.01it/s, batch_loss=0.6062]


✨ Epoch 2 Completed! Avg Loss: 0.9543


Training Epoch 3: 100%|██████████| 8074/8074 [59:56<00:00,  2.25it/s, batch_loss=0.9072] 


✨ Epoch 3 Completed! Avg Loss: 0.8792


In [12]:
model.eval()
val_loss = 0

# Disable gradient calculations to save VRAM and speed up validation
with torch.no_grad():
    val_progress = tqdm(val_loader, desc="Validating Model")
    
    for batch in val_progress:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        logits = model(images, input_ids, attention_mask)
        
        # Shift tokens for causal language loss calculation
        shift_logits = logits[:, 1:-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        
        loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        val_loss += loss.item()
        
        val_progress.set_postfix({"val_batch_loss": f"{loss.item():.4f}"})

final_val_loss = val_loss / len(val_loader)
print(f"\n✨ Validation Complete! Average Validation Loss: {final_val_loss:.4f}")


# --- 2. FINAL SAVE SNAPSHOT ---
FINAL_CHECKPOINT_PATH = f"mimic_vlm_epoch{epoch}_FINAL.pt"

torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': epoch_loss / len(train_loader),
    'val_loss': final_val_loss
}, FINAL_CHECKPOINT_PATH)

print(f"💾 Absolute final checkpoint saved successfully to: {FINAL_CHECKPOINT_PATH}")
print("🎉 All tasks completed successfully. You can safely go to sleep now!")

Validating Model: 100%|██████████| 63/63 [00:16<00:00,  3.75it/s, val_batch_loss=1.4923]



✨ Validation Complete! Average Validation Loss: 0.9423
💾 Absolute final checkpoint saved successfully to: mimic_vlm_epoch3_FINAL.pt
🎉 All tasks completed successfully. You can safely go to sleep now!


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Initialize a clean instance of your model architecture
model = MedicalReportGenerator().to(device)

# 2. Provide the exact path to the file you just uploaded under Kaggle's input section
UPLOADED_MODEL_PATH = "/kaggle/input/models/muhammad0nur/mimic-vlm-epoch3-final-pt/pytorch/default/1/mimic_vlm_epoch3_FINAL.pt"

# 3. Load the weights cleanly
checkpoint = torch.load(UPLOADED_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("🎯 Successfully loaded your joint fine-tuned model weights! Ready for evaluation.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🎯 Successfully loaded your joint fine-tuned model weights! Ready for evaluation.


In [14]:
# 1. Switch model to evaluation mode (freezes batchnorm running metrics)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def generate_medical_report(image_path, max_generation_length=64):
    """
    Takes a path to a chest X-ray image, processes it through the trained
    DenseNet-GPT2 pipeline, and generates a predicted clinical text report.
    """
    # Load and transform the target image safely
    try:
        raw_image = Image.open(image_path).convert('RGB')
        # Uses the exact transformation pipeline from your dataset definition
        image_tensor = train_dataset.transform(raw_image).unsqueeze(0).to(device)
    except Exception as e:
        return f"Error loading image: {str(e)}"
    
    # Extract visual prefix tokens using the newly trained model
    with torch.no_grad():
        # Step A: Extract image features from DenseNet
        visual_features = model.encoder(image_tensor)
        # Step B: Map feature dimensions through your projection layer [1024 -> 768]
        visual_embeddings = model.projector(visual_features).unsqueeze(1) 
        
        # Initialize token tracking array with GPT-2 Bos Token or first text seed
        generated_tokens = [tokenizer.bos_token_id if tokenizer.bos_token_id is not None else 50256]
        
        # Autoregressive decoding loop
        for _ in range(max_generation_length):
            input_ids_tensor = torch.tensor([generated_tokens]).to(device)
            
            # Combine the physical visual embedding prefix with text tokens generated so far
            text_embeddings = model.decoder.transformer.wte(input_ids_tensor)
            inputs_embeds = torch.cat((visual_embeddings, text_embeddings), dim=1)
            
            # Forward pass through GPT-2 decoder layers
            outputs = model.decoder(inputs_embeds=inputs_embeds)
            next_token_logits = outputs.logits[:, -1, :]
            
            # Greedy choice: Pick the highest probability token mapping
            next_token = torch.argmax(next_token_logits, dim=-1).item()
            generated_tokens.append(next_token)
            
            # Stop predicting immediately if the network outputs the End-of-Text tag
            if next_token == tokenizer.eos_token_id:
                break
                
        # Decode numbers back into real human-readable medical sentences
        predicted_report = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        return predicted_report

# --- RUN A SANITY TEST REPORT ---
# Grab an actual validation sample path from your CSV to see how it performs!
sample_row = val_dataset.df.iloc[0]
import ast
sample_paths = ast.literal_eval(sample_row['image'])
full_test_path = Path("/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/official_data_iccv_final") / sample_paths[0]

print("🔍 Target Image File Path:", full_test_path)
print("📝 Ground Truth Label (What the Doctor wrote):")
print(ast.literal_eval(sample_row['text'])[0])

print("\n🤖 AI Predicted Medical Report:")
generated_result = generate_medical_report(full_test_path)
print(generated_result)

🔍 Target Image File Path: /kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/official_data_iccv_final/files/p10/p10003502/s50084553/70d7e600-373c1311-929f5ff9-23ee3621-ff551ff9.jpg
📝 Ground Truth Label (What the Doctor wrote):
Findings:  Impression: Compared to chest radiographs since ___, most recently ___.  Large right and moderate left pleural effusions and severe bibasilar atelectasis are unchanged.  Cardiac silhouette is obscured.  No pneumothorax.  Pulmonary edema is mild, obscured radiographically by overlying abnormalities.

🤖 AI Predicted Medical Report:
 in the chest is not well evaluated.  ET tube is in standard position.  NG tube tip is in the stomach.  Right IJ central line tip is in the mid SVC.  There is no pneumothorax.  There is a small right pleural effusion.  There is mild pulmonary


In [16]:
# 1. Force GPU targeting
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Define a function to unfreeze the layers for joint training
def prepare_model_for_joint_tuning(model):
    # Unfreeze all parameters across the entire architecture
    for param in model.parameters():
        param.requires_grad = True
        
    print("🔓 All layers (DenseNet121 + Projector + GPT2) are now UNFROZEN.")
    
    # Return all parameters to the optimizer
    return model.parameters()

# 3. Unfreeze your active model instance
all_trainable_params = prepare_model_for_joint_tuning(model)

# 4. Initialize a new optimizer with a highly conservative learning rate
# Using 2e-5 prevents the text decoder weights from exploding
optimizer = torch.optim.AdamW(all_trainable_params, lr=2e-5)

print("🚀 Phase 2 Optimizer initialized. Ready to begin Joint Fine-Tuning!")

🔓 All layers (DenseNet121 + Projector + GPT2) are now UNFROZEN.
🚀 Phase 2 Optimizer initialized. Ready to begin Joint Fine-Tuning!


In [17]:
# Ensure configurations are locked down
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

# Configuration for Phase 2 Run
TOTAL_JOINT_EPOCHS = 3
START_EPOCH = 3  # Picking up as Epoch 3 following your previous checkpoint configurations

print(f"🌙 Starting Phase 2 Joint Fine-Tuning from Epoch {START_EPOCH} for {TOTAL_JOINT_EPOCHS} iterations...")

for epoch in range(START_EPOCH, START_EPOCH + TOTAL_JOINT_EPOCHS):
    # =================================================================
    # STEP 1: TRAINING PHASE (Gradient Tracking Enabled)
    # =================================================================
    model.train()
    train_loss = 0.0
    
    train_progress = tqdm(train_loader, desc=f"🎬 Joint Train Epoch {epoch}")
    for batch in train_progress:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass updates the weights of both the vision and text networks simultaneously
        logits = model(images, input_ids, attention_mask)
        
        # Shift tokens for autoregressive causal language calculation
        shift_logits = logits[:, 1:-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        
        loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_progress.set_postfix({"train_batch_loss": f"{loss.item():.4f}"})
        
    avg_train_loss = train_loss / len(train_loader)
    print(f"\n📊 Epoch {epoch} Training Summary -> Avg Loss: {avg_train_loss:.4f}")
    
    # =================================================================
    # STEP 2: VALIDATION PHASE (Evaluation Mode, Gradients Disabled)
    # =================================================================
    model.eval()
    val_loss = 0.0
    
    val_progress = tqdm(val_loader, desc=f"🧪 Joint Val Epoch {epoch}")
    with torch.no_grad():
        for batch in val_progress:
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(images, input_ids, attention_mask)
            
            shift_logits = logits[:, 1:-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            
            loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            val_loss += loss.item()
            val_progress.set_postfix({"val_batch_loss": f"{loss.item():.4f}"})
            
    avg_val_loss = val_loss / len(val_loader)
    print(f"📉 Epoch {epoch} Validation Summary -> Avg Loss: {avg_val_loss:.4f}")
    
    # =================================================================
    # STEP 3: PERSISTENT CLOUD CHECKPOINT SAVING
    # =================================================================
    CHECKPOINT_NAME = f"mimic_vlm_joint_epoch{epoch}_checkpoint.pt"
    
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss
    }, CHECKPOINT_NAME)
    
    print(f"💾 Checkpoint safely backed up to cloud disk: {CHECKPOINT_NAME}\n" + "="*60)

print("🎉 Phase 2 Joint Fine-Tuning completely finished! All models saved to /kaggle/working.")

🌙 Starting Phase 2 Joint Fine-Tuning from Epoch 3 for 3 iterations...


🎬 Joint Train Epoch 3: 100%|██████████| 8074/8074 [1:14:09<00:00,  1.81it/s, train_batch_loss=1.0209]



📊 Epoch 3 Training Summary -> Avg Loss: 0.7822


🧪 Joint Val Epoch 3: 100%|██████████| 63/63 [00:13<00:00,  4.81it/s, val_batch_loss=1.4905]


📉 Epoch 3 Validation Summary -> Avg Loss: 0.9196
💾 Checkpoint safely backed up to cloud disk: mimic_vlm_joint_epoch3_checkpoint.pt


...

📉 Epoch 4 Validation Summary -> Avg Loss: 0.9216
💾 Checkpoint safely backed up to cloud disk: mimic_vlm_joint_epoch4_checkpoint.pt


🎬 Joint Train Epoch 5: 100%|██████████| 8074/8074 [1:02:52<00:00,  2.14it/s, train_batch_loss=0.2431]



📊 Epoch 5 Training Summary -> Avg Loss: 0.7371


🧪 Joint Val Epoch 5: 100%|██████████| 63/63 [00:10<00:00,  6.23it/s, val_batch_loss=1.5517]


📉 Epoch 5 Validation Summary -> Avg Loss: 0.9271
💾 Checkpoint safely backed up to cloud disk: mimic_vlm_joint_epoch5_checkpoint.pt
🎉 Phase 2 Joint Fine-Tuning completely finished! All models saved to /kaggle/working.


In [18]:
import os
from IPython.display import FileLink

print("✨ All training epochs completed! Starting safe automated teardown...")

# 1. Final comprehensive model save
FINAL_MODEL_PATH = "mimic_vlm_phase2_fully_trained.pt"

torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'final_train_loss': avg_train_loss if 'avg_train_loss' in locals() else "completed",
    'final_val_loss': avg_val_loss if 'avg_val_loss' in locals() else "completed"
}, FINAL_MODEL_PATH)

print(f"💾 Absolute final weights permanently locked to: {FINAL_MODEL_PATH}")

# 2. Force an immediate browser download link
if os.path.exists(FINAL_MODEL_PATH):
    print("\n👇 CLICK THE LINK BELOW TO DOWNLOAD TO YOUR MACHINE BEFORE CLOSING THE TAB 👇")
    display(FileLink(FINAL_MODEL_PATH))
else:
    print("❌ Critical error: Final model file was not found on disk.")

✨ All training epochs completed! Starting safe automated teardown...
💾 Absolute final weights permanently locked to: mimic_vlm_phase2_fully_trained.pt

👇 CLICK THE LINK BELOW TO DOWNLOAD TO YOUR MACHINE BEFORE CLOSING THE TAB 👇


/kaggle/working/mimic_vlm_phase2_fully_trained.pt

In [2]:

# 1. Ensure model is in evaluation mode
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def generate_final_report(image_path, max_generation_length=90):
    """
    Inference pipeline using greedy decoding to generate text 
    from the fully fine-tuned DenseNet121 + GPT-2 model.
    """
    try:
        raw_image = Image.open(image_path).convert('RGB')
        image_tensor = train_dataset.transform(raw_image).unsqueeze(0).to(device)
    except Exception as e:
        return f"Error loading image: {str(e)}"
    
    with torch.no_grad():
        # Feature extraction & dimensionality projection
        visual_features = model.encoder(image_tensor)
        visual_embeddings = model.projector(visual_features).unsqueeze(1) 
        
        # Initialize token tracking with GPT-2 Bos Token ID
        bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else 50256
        generated_tokens = [bos_id]
        
        # Autoregressive generation loop
        for _ in range(max_generation_length):
            input_ids_tensor = torch.tensor([generated_tokens]).to(device)
            text_embeddings = model.decoder.transformer.wte(input_ids_tensor)
            
            # Concatenate visual prefix with predicted text tokens
            inputs_embeds = torch.cat((visual_embeddings, text_embeddings), dim=1)
            
            outputs = model.decoder(inputs_embeds=inputs_embeds)
            next_token_logits = outputs.logits[:, -1, :]
            
            next_token = torch.argmax(next_token_logits, dim=-1).item()
            generated_tokens.append(next_token)
            
            if next_token == tokenizer.eos_token_id:
                break
                
        return tokenizer.decode(generated_tokens, skip_special_tokens=True)

# 2. Run a comparative test on 3 different validation samples
print("🔥 SIMULATING INFERENCE ON FULLY FINE-TUNED PATHWAYS 🔥\n" + "="*70)

for idx in range(3):
    sample_row = val_dataset.df.iloc[idx]
    sample_paths = ast.literal_eval(sample_row['image'])
    
    # Reconstruct the target file path dynamically
    # Update this path string if your input directory layout is structured differently!
    base_dir = Path("/kaggle/input/mimic-cxr-dataset/official_data_iccv_final")
    if not base_dir.exists():
        base_dir = Path("MIMC-CXR/official_data_iccv_final")
        
    full_test_path = base_dir / sample_paths[0]
    
    print(f"\n🔬 [SAMPLE {idx+1}] Image Path: {sample_paths[0]}")
    print("-" * 50)
    print("📝 GROUND TRUTH REPORT (Doctor):")
    print(ast.literal_eval(sample_row['text'])[0])
    print("-" * 50)
    
    # Generate prediction from our freshly trained model
    predicted_text = generate_final_report(full_test_path)
    print("🤖 MODEL PREDICTION:")
    print(predicted_text)
    print("=" * 70)

NameError: name 'model' is not defined